# Drone Tracker — v4 yolov8s training (D4, 2026-04-28)

Trenuje **dwa warianty v4**:
- `yolov8s @ imgsz=640` — speed-first, ~30 ms estimate na NPU XDNA2
- `yolov8s @ imgsz=960` — accuracy-first, kontynuacja v3 podejscia

Wybor zalezy od D5 benchmarku — D4 trenuje oba, D5 mierzy mAP + inference time, D6 demo z lepszym.

**Wymagania**:
- Colab Pro (T4/V100, Free Tier moze nie zdazyc)
- v4 dataset zip (z `tools/_build_v4_dataset.py` po CVAT export)
- ~10-12h sequential albo ~6h parallel na 2 GPU

**Plik wyjscia (Drive)**: `MyDrive/drone_tracker/v4/`
  - `v4_drone_s_imgsz640/best.pt` + `v4_best_fp16_imgsz640.onnx`
  - `v4_drone_s_imgsz960/best.pt` + `v4_best_fp16_imgsz960.onnx`

## Krok 1 — Weryfikacja GPU

In [ ]:
!nvidia-smi

Powinno pokazac T4 (16 GB) lub V100/A100 (Pro). Jesli T4 — batch=32 OK, jesli mniej VRAM zmniejsz batch w cells trening.

## Krok 2 — Instalacja

In [ ]:
!pip install -q ultralytics==8.4.30 onnx onnxslim

In [ ]:
import torch
from ultralytics import YOLO
print(f'torch={torch.__version__}, cuda={torch.cuda.is_available()}, device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')

## Krok 3 — Mount Drive + upload v4 dataset

Zalozenia:
1. Lokalnie po CVAT export odpalilas `python tools/_build_v4_dataset.py <cvat.zip> --include-v3`
2. Powstale `training/v4/` zzipowalas: `zip -r v4_dataset.zip training/v4/`
3. Wgralas na Drive: `MyDrive/drone_tracker/v4_dataset.zip`

Albo szybciej: zostaw v4_dataset.zip na lokalnym dysku i wgraj przez Colab Files panel.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder dla v4 outputs
import os
DRIVE_BASE = '/content/drive/MyDrive/drone_tracker/v4'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive output: {DRIVE_BASE}')

In [ ]:
# Wypakuj v4 dataset (zaloz: v4_dataset.zip na Drive)
import shutil, os
from pathlib import Path

ZIP_PATH = '/content/drive/MyDrive/drone_tracker/v4_dataset.zip'
WORK_DIR = '/content/v4_work'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'Brak {ZIP_PATH}. Wgraj v4_dataset.zip na Drive lub uzyj Files panel.')

os.makedirs(WORK_DIR, exist_ok=True)
shutil.unpack_archive(ZIP_PATH, WORK_DIR)
print(f'Rozpakowano do {WORK_DIR}')
!ls {WORK_DIR}/training/v4/

## Krok 4 — Verify dataset

In [ ]:
import os
DATA_YAML = f'{WORK_DIR}/training/v4/data.yaml'
TRAIN_IMGS = f'{WORK_DIR}/training/v4/images/train'
VAL_IMGS = f'{WORK_DIR}/training/v4/images/val'

print('data.yaml:')
print(open(DATA_YAML).read())

n_train = sum(1 for f in os.listdir(TRAIN_IMGS) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
n_val = sum(1 for f in os.listdir(VAL_IMGS) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print(f'\nimages: train={n_train}  val={n_val}  total={n_train + n_val}')
if n_train < 1000:
    print('WARN: train set < 1000 — moze byc za malo dla v4 yolov8s + 50 epok')

## Krok 5 — Trening v4 yolov8s @ imgsz=640 (speed-first)

ETA T4: ~5-6h dla 50 epok @ 2000 obrazow. V100 ~3-4h.

In [ ]:
from ultralytics import YOLO
import os

RUN_NAME_640 = 'v4_drone_s_imgsz640'
RUN_PROJECT = '/content/runs'

model = YOLO('yolov8s.pt')
model.train(
    data=DATA_YAML,
    imgsz=640,
    epochs=50,
    batch=32,         # T4 16GB OK; zmniejsz do 16 jesli OOM
    device=0,
    patience=15,
    name=RUN_NAME_640,
    project=RUN_PROJECT,
    optimizer='AdamW', lr0=0.001, weight_decay=0.0005,
    warmup_epochs=3.0, cos_lr=True,
    mosaic=1.0, mixup=0.15, scale=0.5, fliplr=0.5,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    close_mosaic=10,
    plots=True, save=True, verbose=True,
)

best_640 = f'{RUN_PROJECT}/{RUN_NAME_640}/weights/best.pt'
print(f'\n[640] best: {best_640}')

## Krok 6 — Eksport ONNX FP32 + FP16 (imgsz=640)

In [ ]:
# FP32
m = YOLO(best_640)
fp32_path = m.export(format='onnx', imgsz=640, opset=12, simplify=False, dynamic=False)
fp32_target = f'{os.path.dirname(best_640)}/v4_best_imgsz640.onnx'
if os.path.exists(fp32_path) and fp32_path != fp32_target:
    os.rename(fp32_path, fp32_target)
print(f'FP32 ONNX: {fp32_target}')

# FP16
m = YOLO(best_640)
fp16_path = m.export(format='onnx', imgsz=640, opset=12, simplify=False, dynamic=False, half=True)
fp16_target = f'{os.path.dirname(best_640)}/v4_best_fp16_imgsz640.onnx'
if os.path.exists(fp16_path) and fp16_path != fp16_target:
    os.rename(fp16_path, fp16_target)
print(f'FP16 ONNX: {fp16_target}')

# Copy do Drive
shutil.copytree(f'{RUN_PROJECT}/{RUN_NAME_640}', f'{DRIVE_BASE}/{RUN_NAME_640}', dirs_exist_ok=True)
print(f'Skopiowano do Drive: {DRIVE_BASE}/{RUN_NAME_640}/')

## Krok 7 — Trening v4 yolov8s @ imgsz=960 (accuracy-first)

ETA T4: ~7-9h dla 50 epok (większy imgsz = wolniej). V100 ~4-5h.

In [ ]:
RUN_NAME_960 = 'v4_drone_s_imgsz960'

model = YOLO('yolov8s.pt')
model.train(
    data=DATA_YAML,
    imgsz=960,
    epochs=50,
    batch=16,         # mniejszy batch dla 960 (kwadratowo wiecej VRAM)
    device=0,
    patience=15,
    name=RUN_NAME_960,
    project=RUN_PROJECT,
    optimizer='AdamW', lr0=0.001, weight_decay=0.0005,
    warmup_epochs=3.0, cos_lr=True,
    mosaic=1.0, mixup=0.15, scale=0.5, fliplr=0.5,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    close_mosaic=10,
    plots=True, save=True, verbose=True,
)

best_960 = f'{RUN_PROJECT}/{RUN_NAME_960}/weights/best.pt'
print(f'\n[960] best: {best_960}')

## Krok 8 — Eksport ONNX FP32 + FP16 (imgsz=960)

In [ ]:
m = YOLO(best_960)
fp32_path = m.export(format='onnx', imgsz=960, opset=12, simplify=False, dynamic=False)
fp32_target = f'{os.path.dirname(best_960)}/v4_best_imgsz960.onnx'
if os.path.exists(fp32_path) and fp32_path != fp32_target:
    os.rename(fp32_path, fp32_target)
print(f'FP32 ONNX: {fp32_target}')

m = YOLO(best_960)
fp16_path = m.export(format='onnx', imgsz=960, opset=12, simplify=False, dynamic=False, half=True)
fp16_target = f'{os.path.dirname(best_960)}/v4_best_fp16_imgsz960.onnx'
if os.path.exists(fp16_path) and fp16_path != fp16_target:
    os.rename(fp16_path, fp16_target)
print(f'FP16 ONNX: {fp16_target}')

shutil.copytree(f'{RUN_PROJECT}/{RUN_NAME_960}', f'{DRIVE_BASE}/{RUN_NAME_960}', dirs_exist_ok=True)
print(f'Skopiowano do Drive: {DRIVE_BASE}/{RUN_NAME_960}/')

## Krok 9 — Eval comparison

Porownanie mAP@0.5 i mAP@0.5:0.95 obu wariantow. Decyzja D5 ktory model na demo.

In [ ]:
for run_name, imgsz in [(RUN_NAME_640, 640), (RUN_NAME_960, 960)]:
    best = f'{RUN_PROJECT}/{run_name}/weights/best.pt'
    if not os.path.exists(best):
        print(f'SKIP {run_name}: brak {best}')
        continue
    m = YOLO(best)
    print(f'\n=== Eval {run_name} @ imgsz={imgsz} ===')
    metrics = m.val(data=DATA_YAML, imgsz=imgsz, batch=8, plots=False, verbose=False)
    print(f'  mAP@0.5      = {metrics.box.map50:.4f}')
    print(f'  mAP@0.5:0.95 = {metrics.box.map:.4f}')
    print(f'  precision    = {metrics.box.mp:.4f}')
    print(f'  recall       = {metrics.box.mr:.4f}')

## Krok 10 — Pobierz weights lokalnie

Po skopiowaniu na Drive (kroki 6 + 8), na lokalnym kompie:
```bash
# Skopiuj z Drive na lokal (np. przez rclone albo recznie)
cp -r ~/Drive/drone_tracker/v4/v4_drone_s_imgsz640 data/weights/v4_640/
cp -r ~/Drive/drone_tracker/v4/v4_drone_s_imgsz960 data/weights/v4_960/
```

Test w C++ pipeline:
```bash
./cpp/build/Release/dtracker_main.exe \
  --video artifacts/test_videos/video_test_wide_short.mp4 \
  --model data/weights/v4_640/v4_best_fp16_imgsz640.onnx \
  --imgsz 640 --no-gui
```

D5 plan: porownaj telemetria z `tools/compare_telemetry.py` v4_640 vs v4_960 vs v3_baseline.